In [1]:
from pathlib import Path

import json
import random

import numpy as np
import pandas as pd

import tensorflow as tf

from tensorflow.keras import Model
from tensorflow.keras.layers import (
    GlobalAveragePooling2D,
    Dropout,
    BatchNormalization,
    Dense,
)
from tensorflow.keras.applications import MobileNet
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.callbacks import (
    ModelCheckpoint,
    EarlyStopping,
    CSVLogger,
)


SEED = 12345

IMAGE_SIZE = (128, 128)
BATCH_SIZE = 32
NUM_CLASSES = 35

FROZEN_EPOCHS = 200
FROZEN_LEARNING_RATE = 1e-4
FROZEN_PATIENCE = 50


random.seed(SEED)
np.random.seed(SEED)
tf.random.set_seed(SEED)


print("TensorFlow version:", tf.__version__)
print("GPU devices:", tf.config.list_physical_devices("GPU"))
print("Image size:", IMAGE_SIZE)
print("Batch size:", BATCH_SIZE)
print("Expected classes:", NUM_CLASSES)

TensorFlow version: 2.20.0
GPU devices: [PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU')]
Image size: (128, 128)
Batch size: 32
Expected classes: 35


In [2]:
KAGGLE_INPUT = Path("/kaggle/input")


MANIFEST_PATH = next(
    KAGGLE_INPUT.rglob("zoolake_clean_split_manifest.csv")
)

CLASS_NAMES_PATH = next(
    KAGGLE_INPUT.rglob("zoolake_class_names.json")
)

DATASET_ROOT = next(
    KAGGLE_INPUT.rglob("zooplankton_0p5x")
)


manifest_df = pd.read_csv(MANIFEST_PATH)

with open(
    CLASS_NAMES_PATH,
    "r",
    encoding="utf-8",
) as file:
    CLASS_NAMES = json.load(file)


manifest_df["filepath"] = manifest_df[
    "relative_filepath"
].apply(
    lambda path: str(DATASET_ROOT / path)
)


train_df = manifest_df[
    manifest_df["split"] == "train"
].copy()

validation_df = manifest_df[
    manifest_df["split"] == "validation"
].copy()

test_df = manifest_df[
    manifest_df["split"] == "test"
].copy()


print("Manifest:", MANIFEST_PATH)
print("Dataset:", DATASET_ROOT)

print("\nTraining:", len(train_df))
print("Validation:", len(validation_df))
print("Test:", len(test_df))
print("Classes:", len(CLASS_NAMES))

Manifest: /kaggle/input/datasets/petroskontos/zoolake-clean-split/zoolake_clean_split_manifest.csv
Dataset: /kaggle/input/datasets/petroskontos/zoolake-full/zooplankton_0p5x

Training: 12558
Validation: 2691
Test: 2691
Classes: 35


In [3]:
train_datagen = ImageDataGenerator(
    rescale=1.0 / 255.0,
    rotation_range=180,
    horizontal_flip=True,
    vertical_flip=True,
    zoom_range=0.20,
    shear_range=10,
)

eval_datagen = ImageDataGenerator(
    rescale=1.0 / 255.0,
)


train_generator = train_datagen.flow_from_dataframe(
    dataframe=train_df,
    x_col="filepath",
    y_col="label",
    classes=CLASS_NAMES,
    target_size=IMAGE_SIZE,
    batch_size=BATCH_SIZE,
    class_mode="categorical",
    color_mode="rgb",
    shuffle=True,
    seed=SEED,
    interpolation="lanczos",
)


validation_generator = eval_datagen.flow_from_dataframe(
    dataframe=validation_df,
    x_col="filepath",
    y_col="label",
    classes=CLASS_NAMES,
    target_size=IMAGE_SIZE,
    batch_size=BATCH_SIZE,
    class_mode="categorical",
    color_mode="rgb",
    shuffle=False,
    interpolation="lanczos",
)


test_generator = eval_datagen.flow_from_dataframe(
    dataframe=test_df,
    x_col="filepath",
    y_col="label",
    classes=CLASS_NAMES,
    target_size=IMAGE_SIZE,
    batch_size=BATCH_SIZE,
    class_mode="categorical",
    color_mode="rgb",
    shuffle=False,
    interpolation="lanczos",
)

Found 12558 validated image filenames belonging to 35 classes.
Found 2691 validated image filenames belonging to 35 classes.
Found 2691 validated image filenames belonging to 35 classes.


In [4]:
backbone = MobileNet(
    weights="imagenet",
    include_top=False,
    input_shape=(128,128, 3),
)

backbone.trainable = False


x = GlobalAveragePooling2D()(backbone.output)

x = Dropout(0.2)(x)
x = BatchNormalization()(x)

x = Dense(
    950,
    activation="relu",
    bias_initializer="zeros",
)(x)

x = Dropout(0.1)(x)
x = BatchNormalization()(x)

outputs = Dense(
    NUM_CLASSES,
    activation="softmax",
    kernel_initializer="random_uniform",
    bias_initializer="zeros",
)(x)


model = Model(
    inputs=backbone.input,
    outputs=outputs,
)


model.compile(
    optimizer=tf.keras.optimizers.Adam(
        learning_rate=FROZEN_LEARNING_RATE
    ),
    loss="categorical_crossentropy",
    metrics=[
        "accuracy",
        tf.keras.metrics.TopKCategoricalAccuracy(
            k=2,
            name="top2_accuracy",
        ),
    ],
)


model.summary()

I0000 00:00:1786530446.117257      23 gpu_device.cc:2020] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 15511 MB memory:  -> device: 0, name: Tesla P100-PCIE-16GB, pci bus id: 0000:00:04.0, compute capability: 6.0


17225924/17225924 ━━━━━━━━━━━━━━━━━━━━ 2s 0us/step


Model: "functional"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ input_layer (InputLayer)        │ (None, 128, 128, 3)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv1 (Conv2D)                  │ (None, 64, 64, 32)     │           864 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv1_bn (BatchNormalization)   │ (None, 64, 64, 32)     │           128 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv1_relu (ReLU)               │ (None, 64, 64, 32)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv_dw_1 (DepthwiseConv2D)     │ (None, 64, 64, 32)     │           288 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv_dw_1_bn                    │ (None, 64, 64, 32)     │           128 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv_dw_1_relu (ReLU)           │ (None, 64, 64, 32)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv_pw_1 (Conv2D)              │ (None, 64, 64, 64)     │         2,048 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv_pw_1_bn                    │ (None, 64, 64, 64)     │           256 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv_pw_1_relu (ReLU)           │ (None, 64, 64, 64)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv_pad_2 (ZeroPadding2D)      │ (None, 65, 65, 64)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv_dw_2 (DepthwiseConv2D)     │ (None, 32, 32, 64)     │           576 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv_dw_2_bn                    │ (None, 32, 32, 64)     │           256 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv_dw_2_relu (ReLU)           │ (None, 32, 32, 64)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv_pw_2 (Conv2D)              │ (None, 32, 32, 128)    │         8,192 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv_pw_2_bn                    │ (None, 32, 32, 128)    │           512 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv_pw_2_relu (ReLU)           │ (None, 32, 32, 128)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv_dw_3 (DepthwiseConv2D)     │ (None, 32, 32, 128)    │         1,152 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv_dw_3_bn                    │ (None, 32, 32, 128)    │           512 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv_dw_3_relu (ReLU)           │ (None, 32, 32, 128)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv_pw_3 (Conv2D)              │ (None, 32, 32, 128)    │        16,384 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv_pw_3_bn                    │ (None, 32, 32, 128)    │           512 │
│ (BatchNormalization)            │                        │             

 Total params: 4,243,795 (16.19 MB)

 Trainable params: 1,010,983 (3.86 MB)

 Non-trainable params: 3,232,812 (12.33 MB)

In [5]:
OUTPUT_DIR = Path(
    "/kaggle/working/mobilenet_verified_head_frozen"
)

OUTPUT_DIR.mkdir(
    parents=True,
    exist_ok=True,
)


BEST_WEIGHTS_PATH = (
    OUTPUT_DIR / "best_frozen.weights.h5"
)

BEST_MODEL_PATH = (
    OUTPUT_DIR / "best_frozen_model.keras"
)

HISTORY_PATH = (
    OUTPUT_DIR / "frozen_history.csv"
)


callbacks = [
    ModelCheckpoint(
        filepath=str(BEST_WEIGHTS_PATH),
        monitor="val_loss",
        mode="min",
        save_best_only=True,
        save_weights_only=True,
        verbose=1,
    ),

    EarlyStopping(
        monitor="val_loss",
        mode="min",
        patience=FROZEN_PATIENCE,
        restore_best_weights=True,
        verbose=1,
    ),

    CSVLogger(
        str(HISTORY_PATH)
    ),
]


print("Maximum epochs:", FROZEN_EPOCHS)
print("Patience:", FROZEN_PATIENCE)
print("Learning rate:", FROZEN_LEARNING_RATE)
print("Output directory:", OUTPUT_DIR)

Maximum epochs: 200
Patience: 50
Learning rate: 0.0001
Output directory: /kaggle/working/mobilenet_verified_head_frozen


In [6]:
train_generator.reset()
validation_generator.reset()


history = model.fit(
    train_generator,
    validation_data=validation_generator,
    epochs=FROZEN_EPOCHS,
    callbacks=callbacks,
    verbose=2,
)

Epoch 1/200


I0000 00:00:1786530460.721798      68 device_compiler.h:196] Compiled cluster using XLA!  This line is logged at most once for the lifetime of the process.



Epoch 1: val_loss improved from None to 0.64854, saving model to /kaggle/working/mobilenet_verified_head_frozen/best_frozen.weights.h5

Epoch 1: finished saving model to /kaggle/working/mobilenet_verified_head_frozen/best_frozen.weights.h5
393/393 - 120s - 306ms/step - accuracy: 0.6644 - loss: 1.3038 - top2_accuracy: 0.7728 - val_accuracy: 0.8213 - val_loss: 0.6485 - val_top2_accuracy: 0.8993
Epoch 2/200

Epoch 2: val_loss improved from 0.64854 to 0.48473, saving model to /kaggle/working/mobilenet_verified_head_frozen/best_frozen.weights.h5

Epoch 2: finished saving model to /kaggle/working/mobilenet_verified_head_frozen/best_frozen.weights.h5
393/393 - 69s - 176ms/step - accuracy: 0.8007 - loss: 0.7120 - top2_accuracy: 0.8934 - val_accuracy: 0.8592 - val_loss: 0.4847 - val_top2_accuracy: 0.9350
Epoch 3/200

Epoch 3: val_loss improved from 0.48473 to 0.42400, saving model to /kaggle/working/mobilenet_verified_head_frozen/best_frozen.weights.h5

Epoch 3: finished saving model to /kaggl

In [7]:
model.load_weights(BEST_WEIGHTS_PATH)

model.save(BEST_MODEL_PATH)


print("Best weights saved at:")
print(BEST_WEIGHTS_PATH)

print("\nComplete model saved at:")
print(BEST_MODEL_PATH)

print("\nTraining history saved at:")
print(HISTORY_PATH)

Best weights saved at:
/kaggle/working/mobilenet_verified_head_frozen/best_frozen.weights.h5

Complete model saved at:
/kaggle/working/mobilenet_verified_head_frozen/best_frozen_model.keras

Training history saved at:
/kaggle/working/mobilenet_verified_head_frozen/frozen_history.csv


In [8]:
from sklearn.metrics import f1_score


validation_generator.reset()

validation_results = model.evaluate(
    validation_generator,
    return_dict=True,
    verbose=1,
)


validation_generator.reset()

validation_probabilities = model.predict(
    validation_generator,
    verbose=1,
)

validation_predictions = np.argmax(
    validation_probabilities,
    axis=1,
)


validation_macro_f1 = f1_score(
    validation_generator.classes,
    validation_predictions,
    average="macro",
)


print("\nFrozen MobileNet validation results:")

print(
    f"Loss: {validation_results['loss']:.4f}"
)

print(
    f"Accuracy: {validation_results['accuracy']:.4f}"
)

print(
    f"Top-2 accuracy: "
    f"{validation_results['top2_accuracy']:.4f}"
)

print(
    f"Macro-F1: {validation_macro_f1:.4f}"
)

85/85 ━━━━━━━━━━━━━━━━━━━━ 5s 61ms/step - accuracy: 0.9316 - loss: 0.2342 - top2_accuracy: 0.9747
85/85 ━━━━━━━━━━━━━━━━━━━━ 11s 95ms/step

Frozen MobileNet validation results:
Loss: 0.2342
Accuracy: 0.9316
Top-2 accuracy: 0.9747
Macro-F1: 0.8013
